<center>Universidad de los Andes - Maestría en Ingeniería de Sistemas y Computación<br>
Algoritmos de Computación Cuantica <br>
Jose Emanuel Toro Medina<br>
🔗 <b>Repositorio de GitHub:</b> <a href="https://github.com/Jtoro109/Talleres-Cuantica-Andes">https://github.com/Jtoro109/Talleres-Cuantica-Andes</a></center>


# Taller: Centro de Colombia y EE. UU. (Algoritmo Naïve)
Para poder ejecutar el siguiente taller/codigos por favor seguir las siguientes intrucciones/"Tabla de contenidos".
 1. Importación y limpieza de datos (Se leen las bases de datos de excel en las siguientes rutas; bases/DIVIPOLA_Municipios.xlsx y bases/uscounties.xlsx).
 2. Guía para Implementar el Algoritmo Naïve del Círculo Mínimo Contenedor (MEC)
 3. Implementación del Algoritmo Naïve (Sin optimizar)
 4. Pruebas y Ejecución Completa con Naïve

## Representación de puntos con números complejos
Cada municipio/condado se representa como un número complejo:
$$z = \text{longitud} + i \cdot \text{latitud}$$

## Fórmulas de Aritmética Compleja

### 1. Círculo a partir de 2 puntos ($z_1$ y $z_2$)
Si el círculo está definido por el segmento que une dos puntos como su diámetro:
- **Centro ($c$):** El punto medio entre ambos.
  $$c = \frac{z_1 + z_2}{2}$$
- **Radio ($r$):** La mitad de la distancia entre ambos.
  $$r = \frac{|z_1 - z_2|}{2}$$

### 2. Círculo a partir de 3 puntos ($z_1$, $z_2$, y $z_3$)
Para hallar el circuncentro ($c$) y circunradio ($r$) de tres puntos no colineales, trasladamos el origen a $z_1$, definiendo:
$$A = z_2 - z_1 \quad y \quad B = z_3 - z_1$$
El circuncentro trasladado $W$ satisface el sistema:
$$\bar{A}W + A\bar{W} = |A|^2$$
$$\bar{B}W + B\bar{W} = |B|^2$$
Resolviendo para $W$ en términos de operaciones complejas:
$$W = \frac{B|A|^2 - A|B|^2}{B\bar{A} - A\bar{B}}$$
Finalmente, deshacemos la traslación para encontrar el centro real y el radio:
$$c = z_1 + W$$
$$r = |c - z_1|$$

*Nota: Si el denominador $D = B\bar{A} - A\bar{B} = 2i \cdot \text{Im}(B\bar{A})$ es cercano a cero, significa que los puntos son colineales y no definen un círculo único.*

## 1. Importación y Limpieza de Datos

Importante respetar/cambiar las rutas en la que se leen las bases:

**Lectura actual:**

- bases/DIVIPOLA_Municipios.xlsx
- bases/uscounties.xlsx

In [ ]:
import pandas as pd
import numpy as np
import time

def cargar_colombia():
    path_col = 'bases/DIVIPOLA_Municipios.xlsx'
    df_col = pd.read_excel(path_col, skiprows=9)
    df_col.columns = ['dpt_code', 'dpt_name', 'mun_code', 'mun_name', 'type', 'lat', 'lon', 'extra']
    
    df_col = df_col.dropna(subset=['lat', 'lon'])
    df_col = df_col[df_col['lat'] != 'LATITUD']
    
    df_col['lat'] = df_col['lat'].astype(float)
    df_col['lon'] = df_col['lon'].astype(float)
    
    z_colombia = df_col['lon'].values + 1j * df_col['lat'].values
    print(f"Colombia: Cargados {len(z_colombia)} municipios correctamente.")
    return z_colombia, df_col

def cargar_usa():
    path_usa = 'bases/uscounties.xlsx'
    df_usa = pd.read_excel(path_usa)
    
    df_usa = df_usa.dropna(subset=['lat', 'lng'])
    df_usa['lat'] = df_usa['lat'].astype(float)
    df_usa['lng'] = df_usa['lng'].astype(float)
    
    z_usa = df_usa['lng'].values + 1j * df_usa['lat'].values
    print(f"USA: Cargados {len(z_usa)} condados correctamente.")
    return z_usa, df_usa

colombia_points, df_colombia = cargar_colombia()
usa_points, df_usa = cargar_usa()

## 2. IMPLEMENTACIÓN DEL ALGORITMO NAÏVE 

In [ ]:
# --- 3. IMPLEMENTACIÓN DEL ALGORITMO NAÏVE (SIN OPTIMIZAR) ---

def obtener_circuncentro(z1, z2, z3):
    A = z2 - z1
    B = z3 - z1
    num = B * (A * A.conjugate()) - A * (B * B.conjugate())
    den = B * A.conjugate() - A * B.conjugate()
    
    if abs(den) < 1e-12:
        return None, None
    
    W = num / den
    c = z1 + W
    r = abs(c - z1)
    return c, r

def contiene_todos(c, r, puntos):
    tolerancia = 1e-9
    for p in puntos:
        if abs(p - c) > r + tolerancia:
            return False
    return True

def encontrar_mec_naive(puntos):
    n = len(puntos)
    R_min = float('inf')
    c_best = None
    puntos_frontera = []

    for i in range(n):
        for j in range(i + 1, n):
            z1, z2 = puntos[i], puntos[j]
            c = (z1 + z2) / 2.0
            r = abs(z1 - z2) / 2.0
            
            if contiene_todos(c, r, puntos):
                if r < R_min:
                    R_min = r
                    c_best = c
                    puntos_frontera = [z1, z2]

    for i in range(n):
        for j in range(i + 1, n):
            for k in range(j + 1, n):
                z1, z2, z3 = puntos[i], puntos[j], puntos[k]
                c, r = obtener_circuncentro(z1, z2, z3)
                
                if r is not None:
                    if contiene_todos(c, r, puntos):
                        if r < R_min:
                            R_min = r
                            c_best = c
                            puntos_frontera = [z1, z2, z3]
                            
    return c_best, R_min, puntos_frontera

## 3. Pruebas y Ejecución Completa con Naïve

### ⚠️ Advertencia sobre la Complejidad Temporal

Este algoritmo naive explora todas las combinaciones:
- Parejas: $\binom{N}{2} \approx O(N^2)$ operaciones de validación.
- Ternas: $\binom{N}{3} \approx O(N^3)$ combinaciones. Cada una requiere revisar los $N$ puntos, resultando en un costo total de **$O(N^4)$**.

Para **Colombia** ($N = 1122$), la ejecución completa tomaría aproximadamente $2.35 \times 10^8$ evaluaciones en el bucle principal, lo cual puede tardar **varias horas** en Python estándar.
Para **EE. UU.** ($N = 3144$), tomaría más de $5 \times 10^9$ evaluaciones.

In [ ]:
import time

# Tomamos una muestra de los primeros n municipios para una prueba rápida
puntos_prueba = colombia_points[:100]

t0 = time.time()
centro, radio, frontera = encontrar_mec_naive(puntos_prueba)
t1 = time.time()

print(f"Prueba completada en: {t1 - t0:.4f} segundos.")
print(f"Centro encontrado: {centro}")
print(f"Radio: {radio}")
print(f"Puntos en la frontera (complejos): {frontera}")

# Descomentar para hacer la prueba total del algoritmo

'''t0 = time.time()
centro_col, radio_col, frontera_col = encontrar_mec_naive(colombia_points)
t1 = time.time()
print(f"Colombia completa resuelta en: {t1 - t0:.4f} segundos.")'''